In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model= init_chat_model("gpt-4o")
standard_model= init_chat_model("gpt-4o-mini")

@wrap_model_call
def state_based_model(request: ModelRequest, handler: callable[[ModelRequest], ModelResponse]) ->ModelResponse:
    """Select model based on State conversation length."""
    messages_count=len(request.messages)

    if messages_count >10:
        model=large_model

    else:
        model=standard_model

    request=request.override(model= model) 
    return handler(request)       

In [3]:
from langchain.agents import create_agent

agent= create_agent(
    model="gpt-4o-mini",
    middleware= [state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."

)

In [4]:
from langchain.messages import HumanMessage

response= agent.invoke(
    {
        "messages":[HumanMessage(content="Did you water the office plant today?")]
    }
)
print(response["messages"][-1].content)

I haven't actually watered the office plant myself, but I can remind you or someone else to take care of it! Would you like me to set a reminder or help with something else related to the plants?


In [5]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-4o-mini-2024-07-18


In [6]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

You should consider repotting once a year, or when you notice the roots starting to grow out of the drainage holes or when the plant's growth slows down. The best time to repot is generally in the spring.


In [7]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-4o-2024-08-06
